# BO Forge simulated campaign

This notebook demonstrates the real MVP v0.1 workflow one experiment at a time.

Each cycle requests one candidate, appends it as `status=suggested`, simulates the lab result, marks that same row as `status=observed`, reloads the CSV log, and only then requests the next candidate.

## 1. Setup

The example config still has `batch_size: 3`, but this notebook deliberately overrides suggestions to `batch_size=1` so the workflow mirrors a sequential lab campaign.

In [ ]:
from pathlib import Path
import os
import shutil
import sys

import numpy as np
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
mpl_cache = PROJECT_ROOT / ".matplotlib-cache"
mpl_cache.mkdir(exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(mpl_cache))
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from bo_forge import CampaignConfig, append_suggestions, load_campaign_log, mark_observed, suggest_next
from bo_forge.diagnostics import plot_diagnostics, plot_progress
from bo_forge.validation import get_observed_data, validate_campaign_data

In [ ]:
config_path = PROJECT_ROOT / "configs" / "simple_2d.yaml"
seed_log_path = PROJECT_ROOT / "examples" / "simple_2d_campaign_log.csv"
working_log_path = PROJECT_ROOT / "examples" / "simple_2d_working_log.csv"
latest_suggestion_path = PROJECT_ROOT / "examples" / "latest_suggestions.csv"

shutil.copyfile(seed_log_path, working_log_path)

config = CampaignConfig.from_yaml(config_path)
one_at_a_time_batch_size = 1

In [ ]:
def simulated_activity(row) -> float:
    ratio = float(row["precursor_ratio"])
    temperature = float(row["annealing_temperature"])
    smooth_peak = 2.2 - 4.0 * (ratio - 0.62) ** 2
    smooth_peak -= ((temperature - 710.0) / 170.0) ** 2
    small_wave = 0.04 * np.sin(10.0 * ratio)
    small_wave += 0.03 * np.cos(temperature / 55.0)
    return float(smooth_peak + small_wave)


def current_log():
    df = load_campaign_log(working_log_path, config)
    validate_campaign_data(config, df)
    return df


def request_one_candidate():
    df = current_log()
    suggestion = suggest_next(config, df, batch_size=one_at_a_time_batch_size)
    suggestion.to_csv(latest_suggestion_path, index=False)
    append_suggestions(working_log_path, suggestion)
    return suggestion


def enter_simulated_result(suggestion):
    row = suggestion.iloc[0]
    result = simulated_activity(row)
    mark_observed(working_log_path, str(row["row_id"]), result)
    return result

## 2. Load the current campaign log

The campaign starts from two manually observed experiments. The remaining initial-design points will be suggested one at a time using Sobol.

In [ ]:
df = current_log()
df

## 3. Request one initial candidate

In [ ]:
suggestion = request_one_candidate()
suggestion

## 4. Enter the result for that one experiment

In a real campaign, replace `simulated_activity(...)` with the measured lab result, then call `mark_observed(...)` for the same `row_id`.

In [ ]:
result = enter_simulated_result(suggestion)
df = current_log()
print("Entered result:", result)
df.tail(3)

## 5. Repeat the same one-experiment cycle

Each cell below represents one lab iteration: suggest one candidate, run one experiment, enter one result, reload the log.

In [ ]:
suggestion = request_one_candidate()
result = enter_simulated_result(suggestion)
df = current_log()
display(suggestion)
print("Entered result:", result)
df.loc[df["row_id"] == str(suggestion.loc[0, "row_id"])]

In [ ]:
suggestion = request_one_candidate()
result = enter_simulated_result(suggestion)
df = current_log()
display(suggestion)
print("Entered result:", result)
df.loc[df["row_id"] == str(suggestion.loc[0, "row_id"])]

In [ ]:
suggestion = request_one_candidate()
result = enter_simulated_result(suggestion)
df = current_log()
display(suggestion)
print("Entered result:", result)
df.loc[df["row_id"] == str(suggestion.loc[0, "row_id"])]

In [ ]:
suggestion = request_one_candidate()
result = enter_simulated_result(suggestion)
df = current_log()
display(suggestion)
print("Entered result:", result)
df.loc[df["row_id"] == str(suggestion.loc[0, "row_id"])]

In [ ]:
suggestion = request_one_candidate()
result = enter_simulated_result(suggestion)
df = current_log()
display(suggestion)
print("Entered result:", result)
df.loc[df["row_id"] == str(suggestion.loc[0, "row_id"])]

## 6. Now the initial design is complete

Once the number of observed rows reaches `initial_design_size`, the next call fits the GP and uses LogEI because this notebook requests one candidate at a time.

In [ ]:
df = current_log()
len(get_observed_data(config, df)), config.bo.initial_design_size

In [ ]:
suggestion = request_one_candidate()
suggestion

In [ ]:
result = enter_simulated_result(suggestion)
df = current_log()
print("Entered result:", result)
df.loc[df["row_id"] == str(suggestion.loc[0, "row_id"])]

## 7. Diagnostics

In [ ]:
plot_progress(config, df);

In [ ]:
plot_diagnostics(config, df);